### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias


# Proyecto de investigación: Reconocimiento de Actividades Humanas (HAR)con Sensores Inerciales

# Importaciones

In [1]:
include("setup.jl")
include("helpers.jl")
include("wrappers.jl")

# Usamos JLD2 para recuperar el estado exacto del preprocesamiento.
if isfile("datos_procesados.jld2")
    JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval
    println("Datos cargados correctamente.")
    println("   - Registros de entrenamiento: $(nrow(df_trainval))")
    println("   - Estructura de folds recuperada: $(typeof(folds_trainval))")
else
    error("No se encontró 'datos_procesados.jld2'. Ejecuta primero Preprocess.ipynb.")
end

Entorno cargado correctamente. Constantes: SEED=104, N_FEATURES=561
Entorno cargado correctamente. Constantes: SEED=104, N_FEATURES=561
Datos cargados correctamente.
   - Registros de entrenamiento: 9205
   - Estructura de folds recuperada: Vector{Tuple{Vector{Int64}, Vector{Int64}}}


# Función para ejecutar modelos

### Esta función tiene como objetivo sistematizar la evaluación de combinaciones entre técnias de selección de características, reducción de dimensionalidad y clasificadores:
- ### Para que cada técnica aparezca al menos una vez, calculamos el número máximo de iteraciones (usando la lista de los modelos, la más larga) y se asignan filtros y reducciones de forma cíclica.
- ### Es reproducible, ya que convierte los diccionarios en listas ordenadas alfabéticamente.
- ### Añadimos una lógica de checkpoint, para que en caso de que la ejecución se detenga antes de terminar, la reanude desde el error, en vez de volver a empezar. Guardamos las métricas en un CSV.
- ### Extraemos accuracy, f1_score, balanced_accuracy media y por cada fold.

In [ ]:
function run_models(dic_filtros, dic_reducciones, dic_modelos, output_file; 
                    X=X_trainval, y=y_trainval, folds=folds_trainval)
    
    # Convierte diccionarios a listas ordenadas para ejecución determinista
    list_filtros     = sort(collect(dic_filtros), by=x->x[1])
    list_reducciones = sort(collect(dic_reducciones), by=x->x[1])
    list_modelos     = sort(collect(dic_modelos), by=x->x[1])

    # Calcula longitud de cada lista de opciones
    n_filtros = length(list_filtros)
    n_reducciones = length(list_reducciones)
    n_modelos = length(list_modelos)

    # Define el número máximo de iteraciones para cubrir todas las técnicas
    max_iter = max(n_filtros, n_reducciones, n_modelos)
    
    # Comprueba si existe archivo previo para reanudar o iniciar de cero
    if isfile(output_file)
        results_df = CSV.read(output_file, DataFrame)
        # Crea conjunto de identificadores para evitar repetir experimentos
        combinaciones_hechas = Set([
            (string(r.Filter), string(r.Reduction), string(r.Model)) 
            for r in eachrow(results_df)
        ])
    else
        # Inicializa DataFrame vacío con las columnas de métricas
        results_df = DataFrame(
            Filter = String[], Reduction = String[], Model = String[],
            Accuracy_Mean = Float64[], 
            F1_Score = Float64[], 
            B_Accuracy_mean = Float64[],
            B_accuracy_list = String[]
        )
        combinaciones_hechas = Set{Tuple{String, String, String}}()
    end
    
    # Define métricas: Accuracy, F1 y Balanced Accuracy
    measures = [accuracy, multiclass_f1score, balanced_accuracy]

    println("Iniciando ejecución: $max_iter experimentos totales.")

    # Bucle principal iterando cíclicamente
    for i in 1:max_iter
        # Selección modular de componentes para cubrir todas las opciones
        pair_filt = list_filtros[(i - 1) % n_filtros + 1]
        pair_red  = list_reducciones[(i - 1) % n_reducciones + 1]
        pair_mod  = list_modelos[(i - 1) % n_modelos + 1]

        # Desempaqueta nombres y objetos
        filt_name, filt_model = pair_filt
        red_name, red_model   = pair_red
        mod_name, mod_model   = pair_mod

        # Salta iteración si la combinación ya existe
        if (filt_name, red_name, mod_name) in combinaciones_hechas
            continue 
        end

        println("\nEvaluando ($i/$max_iter): [$filt_name] + [$red_name] + [$mod_name]")
        
        # Construye el pipeline con las técnicas seleccionadas
        pipe = PersonalizedPipeline(
            scaler    = MyMinMaxScaler(), 
            filter    = filt_model,      
            reduction = red_model,       
            clf       = mod_model        
        )
        
        try
            # Instancia la 'machine' de MLJ y ejecuta evaluación
            mach = machine(pipe, X, y) 
            evaluation = evaluate!(
                mach, 
                resampling = folds, 
                measures = measures, 
                verbosity = 0,
                acceleration = CPUThreads() 
            )
            
            # Extrae medias de las métricas
            acc_mean = evaluation.measurement[1]
            f1_mean   = evaluation.measurement[2]
            b_acc_mean   = evaluation.measurement[3]
            
            # Obtiene lista de Balanced Accuracy por fold y serializa a string
            b_acc_folds = evaluation.per_fold[3]
            b_acc_str = join(round.(b_acc_folds, digits=5), ";")

            println("Acc: $(round(acc_mean, digits=4)) | F1: $(round(f1_mean, digits=4)) | B_acc: $(round(b_acc_mean, digits=4))")
            
            # Guarda resultados en DataFrame y actualiza CSV
            push!(results_df, (filt_name, red_name, mod_name, acc_mean, f1_mean, b_acc_mean, b_acc_str), promote=true)
            CSV.write(output_file, results_df)
            
        catch e
            # Captura errores para no detener el flujo y registra fallo
            println("ERROR en $filt_name + $red_name + $mod_name: $e")
            push!(results_df, (filt_name, red_name, mod_name, NaN, NaN, NaN, "ERROR"))
            CSV.write(output_file, results_df)
        end
        
        # Libera memoria tras cada iteración
        GC.gc()
    end
    
    println("Experimento finalizado.")
    return results_df
end

run_models (generic function with 1 method)

# Definición de los diccionarios para los modelos básicos y selección de atributos

In [ ]:
# Diccionarios de los filtrados
dic_filtros = Dict(
    "Sin_Filtrado" => nothing,
    "ANOVA" => MyANOVAFilter(n_features=N_FEATURES),
    "Pearson" => MyPearsonFilter(n_features=N_FEATURES),
    "Spearman" => MySpearmanFilter(n_features=N_FEATURES),
    "Kendall" => MyKendallFilter(n_features=N_FEATURES),
    "MI" => MyMIFilter(n_features=N_FEATURES),
    "RFE" => MyRFEFilter(n_features=N_FEATURES)
)

# Diccionarios de las reducciones de dimensionalidad
dic_reducciones = Dict(
    "Sin reducción" => IdentityTransformer(),
    "PCA" => PCA(variance_ratio=0.95), # Conservar el 95% de la varianza
    "ICA" => ICA(outdim=2, maxiter=10000,tol=0.5), # Tolerancia alta para evitar error
    "LDA" => LDA(method=:whiten, outdim=5) # En LDA, la dimensión es n_classes - 1
)

# Diccionarios de modelos
dic_modelos = Dict(
    # MLP
    "NeuralNetwork_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    "NeuralNetwork_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    "NeuralNetwork_100_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))),
    
    # KNN
    "KNN_1" => KNNClassifier(K=1),
    "KNN_10" => KNNClassifier(K=10),
    "KNN_20" => KNNClassifier(K=20),

    # SVM
    "SVM_0.1" => ProbabilisticSVC(cost=0.1),
    "SVM_0.5" => ProbabilisticSVC(cost=0.5),
    "SVM_1.0" => ProbabilisticSVC(cost=1.0)
);

In [ ]:
# Ejecutar y guardar
df_resultados_basicos = run_models(
    dic_filtros, 
    dic_reducciones, 
    dic_modelos, 
    "resultados_modelos_basicos.csv"
);

Iniciando ejecución: 9 experimentos totales.

Evaluando (1/9): [ANOVA] + [ICA] + [KNN_1]
Acc: 0.5917 | F1: 0.5906 | B_acc: 0.5901
Matriz de confusión guardada como: ANOVA + ICA + KNN_1 

Evaluando (2/9): [Kendall] + [LDA] + [KNN_10]
Acc: 0.9575 | F1: 0.9575 | B_acc: 0.9577
Matriz de confusión guardada como: Kendall + LDA + KNN_10 

Evaluando (3/9): [MI] + [PCA] + [KNN_20]
Acc: 0.8941 | F1: 0.8933 | B_acc: 0.8931
Matriz de confusión guardada como: MI + PCA + KNN_20 

Evaluando (4/9): [Pearson] + [Sin reducción] + [NeuralNetwork_100]
Acc: 0.9077 | F1: 0.9051 | B_acc: 0.907
Matriz de confusión guardada como: Pearson + Sin reducción + NeuralNetwork_100 

Evaluando (5/9): [RFE] + [ICA] + [NeuralNetwork_100_50]
Acc: 0.5953 | F1: 0.5597 | B_acc: 0.5932
Matriz de confusión guardada como: RFE + ICA + NeuralNetwork_100_50 

Evaluando (6/9): [Sin_Filtrado] + [LDA] + [NeuralNetwork_50]
Acc: 0.8955 | F1: 0.8906 | B_acc: 0.8989
Matriz de confusión guardada como: Sin_Filtrado + LDA + NeuralNetwork_50

# Definición de los diccionarios de los modelos de ensemble

### Para implementar un SVM lineal, usamos un clasificador SGD configurado para comportarse como un SVM lineal. Esto lo hacemos por el alto coste computacional de la implementación estándar (SVM = O(n**3) Y SGDClassifier = O(n)). Los argumentos nos ayudan a esto:
- ### loss = "hinge": Este parámetro transforma el clasificador genérico en una SVM. La función hinge loss penaliza las predicciones incorrectas y aquellas que, aún siendo correctas, se encuentran dentro del margen de decisión. Busca maximizar el margen de separación entre clases.
- ### penalty = "l2" : Se aplica una regularización l2. Esto evita que los pesos crezcan desmesuradamente, reduciendo el riesgo de overfitting y mejorando la generalización del margen. Es la penalización estándar en las formulaciones SVM clásicas.
- ### alpha = 0.0001 : Controla la intensidad de la penalización l2. Un valor de 0.0001 es el estándar que ofrece un equilibrio entre mantener el modelo simple (evitando varianza alta) y permitir que se ajuste suficientemente a los datos de entrenamiento (evitando sesgo alto).
- ### random_state = SEED : Como SGD es estocástico, fijamos la semilla para garantizar reproducibilidad.

In [ ]:
# No se especifica usar ningún filtrado para ensembles, debemos decirle a nuestra función que no hay
dic_filtros_ensemble = Dict(
    "Sin_Filtrado" => nothing
)

# Usamos PCA con el 95% de la varianza o ninguno
dic_reducciones_ensemble = Dict(
    "Sin_Reduccion" => IdentityTransformer(), 
    "PCA_95"        => PCA(variance_ratio=0.95)
)

# Definición de los modelos base para los ensembles
knn_base = KNNClassifier(K=5)
svm_base = SKSGDClassifier(loss = "hinge", penalty = "l2", alpha = 0.0001, random_state = SEED) 

# Diccionario de los modelos de ensemble
dic_modelos_ensemble = Dict(
    # BaggingClassifier
    "Bagging_KNN_10" => EnsembleModel(model = knn_base, n = 10),
    "Bagging_KNN_50" => EnsembleModel(model = knn_base, n = 50),

    # AdaBoost
    "AdaBoost_SVM" => AdaBoostClassifier(estimator = svm_base, n_estimators = 5, algorithm = "SAMME"),

    # EvoTree
    "EvoTree_50" => EvoTreeClassifier(nrounds = 50, eta = 0.2),
    "EvoTree_100" => EvoTreeClassifier(nrounds = 100, eta = 0.2)
)

Dict{String, Probabilistic} with 5 entries:
  "EvoTree_50"     => EvoTreeClassifier(loss = mlogloss, …)
  "AdaBoost_SVM"   => AdaBoostClassifier(estimator = SGDClassifier(loss = hinge…
  "Bagging_KNN_10" => ProbabilisticEnsembleModel(model = KNNClassifier(K = 5, ……
  "Bagging_KNN_50" => ProbabilisticEnsembleModel(model = KNNClassifier(K = 5, ……
  "EvoTree_100"    => EvoTreeClassifier(loss = mlogloss, …)

In [ ]:
df_resultados_ensembles = run_experiment(
    dic_filtros_ensemble, 
    dic_reducciones_ensemble, 
    dic_modelos_ensemble, 
    "resultados_modelos_ensemble.csv"
)

Archivo de checkpoint encontrado: resultados_ensembles_bagging.csv
1 experimentos completados previamente.

Evaluando: [Sin_Filtrado] + [PCA_95] + [Bagging_KNN_10]
Acc: 0.894 | F1: 0.8941 | B_acc: 0.894

Evaluando: [Sin_Filtrado] + [PCA_95] + [Bagging_KNN_50]
Acc: 0.8931 | F1: 0.8933 | B_acc: 0.8932

Evaluando: [Sin_Filtrado] + [PCA_95] + [EvoTree_100]
Acc: 0.8893 | F1: 0.8882 | B_acc: 0.8888

Evaluando: [Sin_Filtrado] + [PCA_95] + [EvoTree_50]
Acc: 0.877 | F1: 0.8758 | B_acc: 0.8762

Evaluando: [Sin_Filtrado] + [Sin_Reduccion] + [AdaBoost_SVM]


┌ Error: Problem fitting the machine machine(:clf, …). 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:695
┌ Info: Running type checks... 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:700
┌ Info: Type checks okay. 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:704
┌ Error: Problem fitting machine(:clf, …)
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:767
┌ Error: Problem fitting the machine machine(PersonalizedPipeline(scaler = MyMinMaxScaler(), …), …). 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:695


ERROR en Sin_Filtrado + Sin_Reduccion + AdaBoost_SVM:
TaskFailedException

┌ Info: Running type checks... 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:700
┌ Info: Type checks okay. 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:704




    nested task error: Python: InvalidParameterError: The 'estimator' parameter of AdaBoostClassifier must be an object implementing 'fit' and 'predict' or None. Got Julia:
    SGDClassifier(
      loss = "hinge", 
      penalty = "l2", 
      alpha = 0.0001, 
      l1_ratio = 0.15, 
      fit_intercept = true, 
      max_iter = 1000, 
      tol = 0.001, 
      shuffle = true, 
      verbose = 0, 
      epsilon = 0.1, 
      n_jobs = nothing, 
      random_state = 104, 
      learning_rate = "optimal", 
      eta0 = 0.0, 
      power_t = 0.5, 
      early_stopping = false, 
      validation_fraction = 0.1, 
      n_iter_no_change = 5, 
      class_weight = nothing, 
      warm_start = false, 
      average = false) instead.
    Python stacktrace:
     [1] validate_parameter_constraints
       @ sklearn.utils._param_validation C:\Users\Pc\.julia\environments\v1.11\.CondaPkg\.pixi\envs\default\Lib\site-packages\sklearn\utils\_param_validation.py:95
     [2] _validate_params
       @ sk

# Evaluación final sobre el conjunto de test

In [ ]:
# Unificamos diccionarios
all_filtros     = merge(dic_filtros, dic_filtros_basico)
all_reducciones = merge(dic_reducciones, dic_reducciones_ensemble)
all_modelos     = merge(dic_modelos, dic_modelos_ensemble)

In [ ]:
# Mejor KNN
best_knn = get_better("KNN", "resultados_modelos_basicos.csv")

# Mejor SVM
best_svm = get_better("SVM", "resultados_modelos_basicos.csv")

# Mejor MLP 
best_mlp = get_better("NeuralNetwork", "resultados_modelos_basicos.csv")

# Mejor Bagging 
best_bagging = get_better("Bagging", "resultados_ensembles_bagging.csv")

# Mejor AdaBoost
best_adaboost = get_better("AdaBoost", "resultados_ensembles_bagging.csv")

In [ ]:
# --- DEFINICIÓN DE MODELOS FINALES ---

# 1. Stacking Ensemble
# "MLP como clasificador final; base SVM + KNN + MLP"
stacking_model = Stack(
    metalearner = NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))), # Juez final
    resampling  = CV(nfolds=3), # Validación interna para entrenar al juez
    model1      = best_svm,     # Campeón 1
    model2      = best_knn,     # Campeón 2
    model3      = best_mlp      # Campeón 3
)

# 2. Hard Voting
# "Con SVM como modelo base" (particionado en 3)
# Nota: MLJ no tiene un "HardVoting" directo que parta datos, pero un EnsembleModel 
# hace votación (soft/hard) sobre el mismo modelo. Si el enunciado pide "particione en 3", 
# suena a un Bagging de 3 estimadores usando el mejor SVM.
voting_model = EnsembleModel(
    model = best_svm.clf, # Ojo: extraemos solo el clasificador del pipeline, o el pipeline entero
    n = 3,
    bagging_fraction = 1.0 # Usar todo el dataset (o particionar si pones < 1.0)
)
# *Nota*: Si prefieres usar los 5 campeones para votar, sería un Vote() distinto. 
# Pero ciñéndonos al texto: "SVM como modelo base".

# 3. Random Forest (Este es fijo)
rf_final = RandomForestClassifier(n_trees=500, max_depth=10)

# 4. Gradient Boosting (Valores por defecto)
xgb_final = XGBoostClassifier() # Requiere cargar librería XGBoost
lgbm_final = LGBMClassifier()   # Requiere cargar librería LightGBM